# Ablation Smoke Test - Path/Extension/Evaluation Validation

**Purpose**: Catch path, extension and evaluation bugs before GPU runs.

**Tests**:
1. Path structure validation (test/images vs test_images)
2. Extension handling (.txt YOLO labels vs .png masks)
3. Evaluation code path (cv2.imread on PNG masks, not YOLO .txt)
4. File existence checks

**Runtime**: <30 seconds on CPU

**Expected Result**: All 3 test images validated, no errors.

In [ ]:
from pathlib import Path

import cv2
import numpy as np

# CRITICAL PATHS (exact structure as production notebooks)
BASE_INPUT_PATH = Path("/kaggle/input/tiny-wound-seg-smoke-test/data_wound_seg")

# Test structure (NEW CORRECT PATHS)
TEST_IMAGES_DIR = BASE_INPUT_PATH / "test/images"  # NOT test_images!
TEST_MASKS_DIR = BASE_INPUT_PATH / "test/masks"  # PNG for evaluation
TEST_LABELS_DIR = BASE_INPUT_PATH / "test/labels"  # YOLO .txt for training

print("=" * 80)
print("PATH VALIDATION")
print("=" * 80)
print(f"Images dir exists: {TEST_IMAGES_DIR.exists()}")
print(f"Masks dir exists: {TEST_MASKS_DIR.exists()}")
print(f"Labels dir exists: {TEST_LABELS_DIR.exists()}")

if not all([TEST_IMAGES_DIR.exists(), TEST_MASKS_DIR.exists(), TEST_LABELS_DIR.exists()]):
    raise FileNotFoundError(
        "BUG 1: Path structure mismatch! Check BASE_INPUT_PATH and subdirectories."
    )

print("OK Path structure correct!")

In [ ]:
print("\n" + "=" * 80)
print("FILE EXTENSION VALIDATION")
print("=" * 80)

# Get test images
test_images = sorted(TEST_IMAGES_DIR.glob("*.png"))
print(f"Test images found: {len(test_images)}")

validation_results = []

for img_path in test_images:
    image_id = img_path.stem

    # Check PNG mask (for evaluation)
    mask_path = TEST_MASKS_DIR / f"{image_id}.png"  # CORRECT: .png extension

    # Check YOLO label (for training)
    label_path = TEST_LABELS_DIR / f"{image_id}.txt"  # CORRECT: .txt extension

    mask_exists = mask_path.exists()
    label_exists = label_path.exists()

    validation_results.append(
        {"image_id": image_id, "mask_exists": mask_exists, "label_exists": label_exists}
    )

    if not mask_exists:
        raise FileNotFoundError(
            f"BUG 2: Missing mask for {image_id} - check extension (.png not .txt!)"
        )

    if not label_exists:
        raise FileNotFoundError(
            f"BUG 2: Missing label for {image_id} - check extension (.txt not .png!)"
        )

    print(f"  {image_id}: mask=OK label=OK")

print(f"\nOK All {len(test_images)} images have both PNG masks and YOLO labels!")

In [ ]:
print("\n" + "=" * 80)
print("EVALUATION CODE PATH VALIDATION")
print("=" * 80)

# Test actual evaluation code (simulating calculate_segmentation_metrics)
for img_path in test_images:
    image_id = img_path.stem

    # Load PNG mask for evaluation, NOT YOLO .txt label!
    mask_path = TEST_MASKS_DIR / f"{image_id}.png"  # CORRECT path

    # This is where Ablation 1 v1-v3 failed!
    gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if gt_mask is None:
        raise ValueError(
            f"BUG 3: cv2.imread() returned None for {mask_path}!\n"
            f"This means we're trying to read a non-image file (like YOLO .txt label).\n"
            f"FIX: Use test/masks/*.png for evaluation, NOT test/labels/*.txt!"
        )

    # Simulate prediction (random for smoke test)
    pred_mask = np.random.randint(0, 2, gt_mask.shape, dtype=np.uint8) * 255

    # Calculate basic metric to ensure no shape errors
    intersection = np.logical_and(pred_mask > 0, gt_mask > 0).sum()
    union = np.logical_or(pred_mask > 0, gt_mask > 0).sum()
    iou = intersection / union if union > 0 else 0

    print(f"  {image_id}: shape={gt_mask.shape}, IoU={iou:.3f} OK")

print(f"\nOK Evaluation code works! cv2.imread() successfully loaded all PNG masks.")

In [ ]:
print("\n" + "=" * 80)
print("SMOKE TEST SUMMARY")
print("=" * 80)
print("OK PATH STRUCTURE: Correct (test/images, test/masks, test/labels)")
print("OK FILE EXTENSIONS: Correct (.png for masks, .txt for YOLO labels)")
print("OK EVALUATION CODE: Correct (cv2.imread on PNG masks works)")
print("OK FILE EXISTENCE: All 3 test images have masks and labels")
print("\nSMOKE TEST PASSED - Notebook structure is valid!")
print("=" * 80)
print("\nReady to swap tiny-wound-seg-smoke-test → baseline-outputs for production run.")